<a href="https://colab.research.google.com/github/aadhavjawahar-sys/Sentiment_Identifier/blob/main/IMDB_Sentiment_Identifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.models import load_model

In [ ]:
from tensorflow.keras.utils import pad_sequences
max_len=500
(x_train, y_train), (x_test, y_test) = keras.datasets.imdb.load_data()
x_train = pad_sequences(x_train, maxlen=max_len)
x_test = pad_sequences(x_test, maxlen=max_len)

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, GlobalAveragePooling1D, Dense


max_features = 100000
max_len = 500

model = Sequential([
    Embedding(input_dim=max_features, output_dim=32),
    #GlobalAveragePooling1D(),
    keras.layers.SimpleRNN(50),
    Dense(100, activation='relu'),
    Dense(1, activation='sigmoid')
])

model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

782/782 ━━━━━━━━━━━━━━━━━━━━ 115s 142ms/step - accuracy: 0.5628 - loss: 0.6693


In [15]:
model.fit(x_train, y_train,epochs=5)

# Save the model
model.save("my_rnn_model.keras")

Epoch 1/5
166/782 ━━━━━━━━━━━━━━━━━━━━ 1:37 158ms/step - accuracy: 0.8867 - loss: 0.2738

KeyboardInterrupt: 

In [ ]:
model = load_model("my_rnn_model.keras")

In [16]:
import numpy as np
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.preprocessing.sequence import pad_sequences
import gradio as gr

# 1. Configuration parameters matching your model setup
start_char = 1
oov_char = 2
index_from = 3
max_features = 100000
max_len = 500

# 2. Reconstruct and compile your model architecture
# (Or replace this block with: model = keras.models.load_model('your_model.h5'))
model = keras.Sequential([
    keras.layers.Embedding(input_dim=max_features, output_dim=32),
    keras.layers.SimpleRNN(50),
    keras.layers.Dense(100, activation='relu'),
    keras.layers.Dense(1, activation='sigmoid')
])
model.compile(optimizer='adam', loss='binary_crossentropy', metrics=['accuracy'])

# 3. Fetch the word index mapping
print("Loading IMDB word index...")
word_index = keras.datasets.imdb.get_word_index()

def text_to_imdb_sequence(text: str) -> np.ndarray:
    """Converts raw text into a padded sequence suitable for model input."""
    if not text.strip():
        return np.zeros((1, max_len), dtype=np.int32)

    # Simple cleanup and tokenization
    words = text.lower().replace(".", "").replace(",", "").replace("!", "").replace("?", "").split()

    sequence = [start_char]
    for word in words:
        if word in word_index:
            idx = word_index[word] + index_from
            if idx < max_features:
                sequence.append(idx)
            else:
                sequence.append(oov_char)
        else:
            sequence.append(oov_char)

    # Pad the sequence to length 500
    padded = pad_sequences([sequence], maxlen=max_len, padding="pre")
    return padded

def predict_sentiment(review_text: str):
    """Predicts whether the input review is positive or negative."""
    if not review_text.strip():
        return "Please enter a review.", {"Positive": 0.0, "Negative": 0.0}

    # Preprocess text into shape (1, 500)
    input_sequence = text_to_imdb_sequence(review_text)

    # Run prediction
    prediction = model.predict(input_sequence)[0][0]  # Value between 0 and 1

    positive_score = float(prediction)
    negative_score = float(1.0 - prediction)

    label = "Positive 😊" if positive_score >= 0.5 else "Negative 😞"

    return label, {"Positive": positive_score, "Negative": negative_score}

# 4. Gradio Interface
app = gr.Interface(
    fn=predict_sentiment,
    inputs=gr.Textbox(
        lines=5,
        placeholder="Type your movie review here...",
        label="Movie Review Text"
    ),
    outputs=[
        gr.Textbox(label="Predicted Sentiment"),
        gr.Label(label="Confidence Scores")
    ],
    title="IMDB Movie Review Sentiment Classifier",
    description="Enter a movie review to predict whether it is positive or negative using your SimpleRNN model."
)

if __name__ == "__main__":
    app.launch()

Loading IMDB word index...
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9bde3094b523fc29bc.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
